# 공통 검증을 사용하는 X-ray 모델 학습
Faster R-CNN → D-FINE-S를 T4에서 순차 실행합니다. 각 모델은 전체 혼합 2,453장·20epoch·512 입력을 사용합니다. 검증 107장으로 매 epoch 저장본을 비교합니다. 테스트는 실행하지 않습니다.

중단 후 T4를 다시 연결하고 위에서부터 실행합니다. Drive에 저장된 완료 epoch부터 복구합니다. GPU 세션 자동 재연결을 보장하지 않습니다. 이 정리 노트는 실제 실행한 코드를 재실행하기 쉽게 정돈한 것입니다. 원격의 최초 환경 설치·GPU 재개 검사는 실행했으나, 새 VM에서 이 정리본 전체를 다시 실행하지는 않았습니다.

## 1. Drive 연결과 실행 환경

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=120000)
import os, sys, subprocess, json, hashlib, shutil, zipfile, time
from pathlib import Path
os.environ['MPLBACKEND']='Agg'
subprocess.run(['nvidia-smi','-L'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
subprocess.run(['uv','python','install','3.12.14'],check=True)
ENV=Path('/content/xray_common_env312')
if not (ENV/'bin/python').exists(): subprocess.run(['uv','venv',str(ENV),'--python','3.12.14'],check=True)
PY=str(ENV/'bin/python')
subprocess.run(['uv','pip','install','--python',PY,'torch==2.8.0','torchvision==0.23.0','--index-url','https://download.pytorch.org/whl/cu126'],check=True)


## 2. 저장된 학습 묶음을 검증하고 복원
첫 실행에만 ZIP 업로드가 필요합니다. 이후에는 Drive 백업을 사용합니다. 기존 ZIP의 SHA-256은 아래 값으로 고정되어 있습니다.

In [ ]:
ROOT=Path('/content/xray_common_project')
STORE=Path('/content/drive/MyDrive/xray_common_epoch_20261004')
assert Path('/content/drive/MyDrive').is_dir()
STORE.mkdir(exist_ok=True)
BUNDLE=Path('/content/xray_common_t4_20261004.zip')
EXPECTED='a47a4d9ad21aebe92d5839f3d0a88e4715ad2c4633f45e7c818d28fac38626dd'
if not BUNDLE.exists():
    if (STORE/BUNDLE.name).exists(): shutil.copy2(STORE/BUNDLE.name,BUNDLE)
    else:
        from google.colab import files
        files.upload()
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest()==EXPECTED
if not (STORE/BUNDLE.name).exists(): shutil.copy2(BUNDLE,STORE/BUNDLE.name)
assert hashlib.sha256((STORE/BUNDLE.name).read_bytes()).hexdigest()==EXPECTED
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(ROOT)
checks=json.loads((ROOT/'bundle_checksums.json').read_text())
assert all(hashlib.sha256((ROOT/n).read_bytes()).hexdigest()==h for n,h in checks.items())
(STORE/'runs').mkdir(exist_ok=True)
if not (ROOT/'runs').exists(): (ROOT/'runs').symlink_to(STORE/'runs',target_is_directory=True)
assert (ROOT/'runs').resolve()==(STORE/'runs').resolve()
subprocess.run(['uv','pip','install','--python',PY,'-r',str(ROOT/'reports/common_epoch_20261004/colab_requirements.txt'),'matplotlib==3.9.4','pycocotools==2.0.11','numpy==1.26.4'],check=True)
subprocess.run([PY,'-c','import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name())'],check=True)
print('Verified files:',len(checks))


## 3. 실제 CUDA·Drive 복구 확인
학습 8장·검증 2장으로 저장과 재개를 확인합니다. 이미 완료한 검사는 저장 상태를 검증하고 통과합니다. 이 작은 자료의 지표는 모델 성능 비교에 사용하지 않습니다.

In [ ]:
PILOT=ROOT/'reports/common_epoch_20261004/cuda_pilot_data'
SOURCE=ROOT/'data/xray_combined_20261003'
for split,count in [('train',8),('val',2)]:
    for kind in ['images','labels']: (PILOT/kind/split).mkdir(parents=True,exist_ok=True)
    for image in sorted((SOURCE/'images'/split).glob('*.png'))[:count]:
        shutil.copy2(image,PILOT/'images'/split/image.name)
        shutil.copy2(SOURCE/'labels'/split/(image.stem+'.txt'),PILOT/'labels'/split/(image.stem+'.txt'))
for name in ['data.yaml','provenance.json']: shutil.copy2(SOURCE/name,PILOT/name)
results=[]
for model in ['faster','dfine']:
    name='v2_common_cuda_pilot_'+model+'_20261004'
    script='xray_dfine.py' if model=='dfine' else 'xray_train_resumable.py'
    cmd=[PY,'scripts/'+script,'--data',str(PILOT),'--name',name,'--epochs','2','--batch','4','--resolution','512','--device','cuda','--resume']
    if model=='faster': cmd+=['--model','faster','--epoch-validation']
    for phase,args in [('pause',['--stop-after-epoch','1']),('resume',[])]:
        with (STORE/(name+'_'+phase+'.log')).open('a') as log:
            result=subprocess.run(cmd+args,cwd=ROOT,stdout=log,stderr=subprocess.STDOUT)
        assert result.returncode==0,(STORE/(name+'_'+phase+'.log')).read_text()[-5000:]
    d=ROOT/'runs'/name
    record=json.loads((d/'execution.json').read_text())
    assert record['status']=='complete' and record['completed_epochs']==2
    assert json.loads((d/'selection.json').read_text())['completed_epochs']==2
    results.append({'model':model,'status':'complete','epochs':2,'seconds':record['seconds']})
(STORE/'cuda_resume_preflight.json').write_text(json.dumps(results,indent=2))
print(results)


## 4. 본학습 시작 또는 재개
기존 세션에서 학습이 실행 중이면 중복 실행하지 않습니다. 같은 실행 환경의 중복 작업은 코드가 차단합니다. 다른 Colab 세션이 동시에 같은 Drive 폴더를 사용하지 않도록 확인합니다. 모델·자료·설정이 달라지면 복구를 거부합니다.

In [ ]:
preflight=json.loads((STORE/'cuda_resume_preflight.json').read_text())
assert {r['model'] for r in preflight if r['status']=='complete'}=={'faster','dfine'}
queue_log=(STORE/'colab_queue.log').open('a')
job=subprocess.Popen([PY,'scripts/xray_common_queue.py','--host','colab','--storage',str(STORE)],cwd=ROOT,stdout=queue_log,stderr=subprocess.STDOUT)
print('TRAINING_QUEUE_STARTED',job.pid,flush=True)
last_message=None
while job.poll() is None:
    qfile=STORE/'queue_colab.json'
    if qfile.exists():
        q=json.loads(qfile.read_text());run=q.get('current')
        progress=STORE/'runs'/str(run)/'progress.json'
        state=json.loads(progress.read_text()) if progress.exists() else {'epoch':'preparing'}
        message={'run':run,'state':q['status'],'progress':state}
        if message!=last_message:
            print(json.dumps(message,ensure_ascii=False),flush=True);last_message=message
    time.sleep(30)
queue_log.close()
assert job.returncode==0,(STORE/'colab_queue.log').read_text()[-6000:]
print('COLAB_TRAINING_COMPLETE; test evaluation has not run')


## 5. 저장 결과 확인
`selection.json`은 검증에서 선택한 epoch·탐지 임계값·지표를 담습니다. `recovery`는 재개용 상태입니다. 실제 정상 제품의 오경보율과 자동 PASS 안전성을 검증한 결과가 아닙니다.

In [ ]:
for model in ['faster','dfine']:
    d=STORE/'runs'/('v2_common20_'+model+'_cuda_20261004')
    for name in ['execution.json','progress.json','selection.json']:
        p=d/name
        if p.exists():
            r=json.loads(p.read_text());r.pop('history',None)
            print(model,name,json.dumps(r,ensure_ascii=False,indent=2))
